#  Парсер kad.arbitr.ru

## Назначение
Сбор данных о судебных делах по ИНН организации с сайта kad.arbitr.ru.

## Требования
- Python 3.8+
- Chrome браузер
- ChromeDriver (соответствующий версии Chrome)Парсер kad.arbitr.ru по ИНН

## Структура
1. Установка зависимостей selenium pandas openpyxl
2. Запуск и роверка зависимостей
3. Ввод ИНН
4. Парсинг данных и сохранение результатов

## Ограничения
- Сайт блокирует IP при частых запросах. Рекомендуется использовать VPN или прокси.
- Если сайт заблокировал вас, то придется увеличивать время выполнения курсоров в коде и менять IP.
- Полный сбор 120 дел занимает 15-20 минут.

## Почему Selenium?

kad.arbitr.ru использует динамическую загрузку данных через JavaScript. Данные об ИНН участников, хронология событий и ссылки на PDF появляются только после кликов и наведения мыши. BeautifulSoup и requests не могут выполнять JavaScript, поэтому для парсинга требуется Selenium с эмуляцией действий пользователя.

## Выходные данные

### Таблица 1
| Столбец | Описание |
|---------|----------|
| ИНН ответчика | ИНН ответчика по делу |
| Наименование ответчика | Название организации-ответчика |
| ИНН истца | ИНН истца по делу |
| Наименование истца | Название организации-истца |
| Номер судебного дела | Номер дела в арбитражном суде |
| Статус судебного разбирательства | Текущий статус дела |
| Системный номер документа | Уникальный идентификатор дела в системе |
| Текст документа | Ссылка на PDF-файл с определением/решением |

### Таблица 2
| Столбец | Описание |
|---------|----------|
| Номер судебного дела | Номер дела |
| Предмет судебного разбирательства | Категория дела |
| Хронология судебного разбирательства | Список событий с датами |
| Сумма требований | Размер исковых требований |
| Сумма возмещения | Размер взыскания (при наличии) |

## Папка с результатами
`parsing_results/` — все файлы сохраняются в эту директорию с временной меткой в имени.

## Установка зависимостей

In [ ]:
pip install selenium
pip install pandas
pip install openpyxl

## Запуск и проверка зависимостей

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time
import pandas as pd
import re
import random
import os
from datetime import datetime

## Ввод ИНН

In [ ]:
choice_INN = input("\nВведите номер ИНН: ").strip()

results_dir = "parsing_results"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
    print(f"Создана папка: {results_dir}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Временная метка: {timestamp}")

%store choice_INN
%store results_dir
%store timestamp


print("Переходите к следующему блоку для запуска парсинга")

# тестовый ИНН 7736323532

## Парсинг данных и сохранение результатов

In [ ]:
%store -r results_dir
%store -r timestamp

if 'results_dir' not in dir() or results_dir is None:
    results_dir = "parsing_results"
if 'timestamp' not in dir() or timestamp is None:
    timestamp = time.strftime("%Y%m%d_%H%M%S")

# Создаем папку для результатов
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

print(f"Формат сохранения: {save_format}")
print(f"Папка результатов: {results_dir}")
print(f"Временная метка: {timestamp}")

def save_dataframes(df1, df2, name_suffix="final"):
    """Сохранение таблиц в формате (Excel)"""
    filename = f"{results_dir}/table_{name_suffix}_{timestamp}"
    
    
    with pd.ExcelWriter(f"{filename}.xlsx") as writer:
        df1.to_excel(writer, sheet_name="Таблица_1_Дела", index=False)
        df2.to_excel(writer, sheet_name="Таблица_2_Детали", index=False)
    print(f"  Сохранено: {filename}.xlsx")
    

# Настройка Chrome для обхода определения автоматизации
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])

driver = webdriver.Chrome(options=options)
driver.get("https://kad.arbitr.ru/")

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#sug-participants > div > textarea")))

# Ввод ИНН и поиск
inn_input = driver.find_element(By.CSS_SELECTOR, "#sug-participants > div > textarea")
inn_input.send_keys(choice_INN) 

search_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
search_button.click()

wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#b-cases tbody tr")))

# ==================== ЭТАП 1: СБОР ДАННЫХ СО ВСЕХ СТРАНИЦ РЕЗУЛЬТАТОВ ====================
all_cases = []
actions = ActionChains(driver)
page = 1

while True:
    print(f"\n=== Сбор данных со страницы {page} ===")
    
    rows = driver.find_elements(By.CSS_SELECTOR, "#b-cases tbody tr")
    print(f"Найдено {len(rows)} дел на странице")
    
    for row in rows:
        try:
            case_link = row.find_element(By.CSS_SELECTOR, "td.num a")
            case_number = case_link.text
            case_url = case_link.get_attribute("href")
            case_id = case_url.split("/")[-1]
            
            # Сбор данных истца через наведение мыши
            plaintiff_inn = None
            plaintiff_name = ""
            try:
                plaintiff_elem = row.find_element(By.CSS_SELECTOR, "td.plaintiff .js-rollover")
                actions.move_to_element(plaintiff_elem).perform()
                time.sleep(0.5)
                tooltips = driver.find_elements(By.CSS_SELECTOR, ".b-rollover:not([style*='display: none'])")
                if tooltips:
                    tooltip_text = tooltips[0].text
                    inn_match = re.search(r'ИНН:\s*(\d+)', tooltip_text)
                    if inn_match:
                        plaintiff_inn = inn_match.group(1)
                    plaintiff_name = tooltip_text.split("\n")[0]
            except:
                pass
            
            # Сбор данных ответчика через наведение мыши
            respondent_inn = None
            respondent_name = ""
            try:
                respondent_elem = row.find_element(By.CSS_SELECTOR, "td.respondent .js-rollover")
                actions.move_to_element(respondent_elem).perform()
                time.sleep(0.5)
                tooltips = driver.find_elements(By.CSS_SELECTOR, ".b-rollover:not([style*='display: none'])")
                if tooltips:
                    tooltip_text = tooltips[0].text
                    inn_match = re.search(r'ИНН:\s*(\d+)', tooltip_text)
                    if inn_match:
                        respondent_inn = inn_match.group(1)
                    respondent_name = tooltip_text.split("\n")[0]
            except:
                pass
            
            all_cases.append({
                "case_number": case_number,
                "case_url": case_url,
                "case_id": case_id,
                "plaintiff_inn": plaintiff_inn if plaintiff_inn else "",
                "plaintiff_name": plaintiff_name if plaintiff_name else "",
                "respondent_inn": respondent_inn if respondent_inn else "",
                "respondent_name": respondent_name if respondent_name else ""
            })
            print(f"  Добавлено дело: {case_number}")
        except:
            continue
    
    # Переход на следующую страницу
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "#pages li.rarr a")
        next_btn.click()
        time.sleep(5)
        page += 1
    except:
        print("\nВсе страницы собраны!")
        break

print(f"\nВсего собрано дел: {len(all_cases)}")

# ==================== ЭТАП 2: СБОР ДЕТАЛЬНОЙ ИНФОРМАЦИИ ИЗ КАРТОЧЕК ДЕЛ ====================
data1 = []
data2 = []

for idx, case in enumerate(all_cases):
    print(f"\nОбработка карточки {idx+1}/{len(all_cases)}: {case['case_number']}")
    
    driver.get(case["case_url"])
    time.sleep(random.uniform(2, 3))
    
    # Статус дела
    status = ""
    try:
        status_elem = driver.find_element(By.CSS_SELECTOR, ".b-case-header-desc")
        status = status_elem.text.strip()
    except:
        pass
    
    # Предмет спора
    subject = ""
    try:
        subject_elem = driver.find_element(By.CSS_SELECTOR, ".b-iblock__header_card span")
        subject = subject_elem.text.strip()
    except:
        try:
            subject_elem = driver.find_element(By.CSS_SELECTOR, ".b-iblock__header_card .b-icon + span")
            subject = subject_elem.text.strip()
        except:
            pass
    
    # Раскрытие всех блоков хронологии (плюсики)
    try:
        expand_btns = driver.find_elements(By.CSS_SELECTOR, ".b-collapse.js-collapse")
        for btn in expand_btns:
            try:
                btn.click()
                time.sleep(0.5)
            except:
                pass
    except:
        pass
    
    # Сумма исковых требований
    claim_amount = ""
    try:
        all_additional = driver.find_elements(By.CSS_SELECTOR, ".additional-info")
        for elem in all_additional:
            text = elem.text
            match = re.search(r'Сумма исковых требований\s+([\d\s,]+\.?\d*)', text)
            if match:
                claim_amount = match.group(1).replace(" ", "")
                break
    except:
        pass
    
    # Ссылка на PDF документа
    document_text = ""
    try:
        pdf_link = driver.find_element(By.CSS_SELECTOR, ".b-chrono-item .b-case-result a[href*='.pdf']")
        document_text = pdf_link.get_attribute("href")
    except:
        pass
    
    # Хронология событий
    chronology = []
    try:
        events = driver.find_elements(By.CSS_SELECTOR, ".b-chrono-item")
        for event in events:
            try:
                date = event.find_element(By.CSS_SELECTOR, ".case-date").text
                event_type = event.find_element(By.CSS_SELECTOR, ".case-type").text
                desc_elem = event.find_element(By.CSS_SELECTOR, ".b-case-result-text")
                desc = desc_elem.text if desc_elem else ""
                chronology.append(f"{date} | {event_type} | {desc}")
            except:
                pass
    except:
        pass
    
    chronology_text = "\n".join(chronology)
    
    # Формирование строки таблицы 1
    data1.append({
        "ИНН ответчика": case["respondent_inn"],
        "Наименование ответчика": case["respondent_name"],
        "ИНН истца": case["plaintiff_inn"],
        "Наименование истца": case["plaintiff_name"],
        "Номер судебного дела": case["case_number"],
        "Статус судебного разбирательства": status,
        "Системный номер документа": case["case_id"],
        "Текст документа": document_text
    })
    
    # Формирование строки таблицы 2
    data2.append({
        "Номер судебного дела": case["case_number"],
        "Предмет судебного разбирательства": subject,
        "Хронология судебного разбирательства": chronology_text,
        "Сумма требований": claim_amount,
        "Сумма возмещения": ""
    })
    
    # Промежуточное сохранение каждые 10 дел
    if (idx + 1) % 10 == 0:
        print(f"  Сохранение промежуточных результатов...")
        temp_df1 = pd.DataFrame(data1)
        temp_df2 = pd.DataFrame(data2)
        if save_format == "excel":
            with pd.ExcelWriter(f"{results_dir}/table1_{timestamp}.xlsx") as writer:
                temp_df1.to_excel(writer, sheet_name="Таблица_1_Дела", index=False)
                temp_df2.to_excel(writer, sheet_name="Таблица_2_Детали", index=False)
        else:
            temp_df1.to_csv(f"{results_dir}/table1_{timestamp}.csv", index=False, encoding="utf-8-sig")
            temp_df2.to_csv(f"{results_dir}/table2_{timestamp}.csv", index=False, encoding="utf-8-sig")

# Финальное сохранение
df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)

save_dataframes(df1, df2, "final")

driver.quit()

# Вывод статистики
print(f"\nВсего обработано дел: {len(all_cases)}")
print(f"Таблица 1: {len(df1)} записей")
print(f"Таблица 2: {len(df2)} записей")
print("\nПервые 5 записей таблицы 1:")
print(df1[["Номер судебного дела", "ИНН истца", "ИНН ответчика", "Статус судебного разбирательства"]].head())